# 01. Image Quality Curation — `proyecto_integrador_v2`

Este notebook corresponde al **Paso 01** del pipeline `proyecto_integrador_v2`.

## Objetivo

Preparar y evaluar la calidad visual de las imágenes que serán usadas para un sistema de **re-identificación visual de perros perdidos y encontrados**.

A diferencia de un proyecto centrado solamente en clasificación de razas, aquí la curaduría debe conservar la mayor cantidad posible de información visual útil para reconocer al mismo perro en distintas fotos: patrones de color, manchas, textura del pelaje, forma de la cara, hocico, orejas, proporciones corporales, postura y marcas visuales particulares.

Por eso, el objetivo de este paso no es transformar agresivamente las imágenes, sino **validarlas, estandarizarlas y medir su calidad**.

## Qué hace este notebook

1. Monta Google Drive.
2. Carga la configuración de `proyecto_integrador_v2`.
3. Busca imágenes en `raw_data/images`, `raw_data/lost_reports`, `raw_data/found_reports` y `raw_data/identity_test`.
4. Valida si las imágenes son legibles.
5. Convierte imágenes a RGB.
6. Calcula métricas de calidad visual.
7. Clasifica la calidad visual de cada imagen.
8. Genera una versión curada ligera.
9. Guarda reportes CSV y visualizaciones.

Para re-identificación visual, evitamos sobreprocesar las imágenes. La curaduría debe mejorar estabilidad técnica, pero no borrar detalles visuales que podrían ayudar a reconocer al mismo perro.

In [ ]:
# 0. Instalación de dependencias

!pip install -q opencv-python pillow pandas numpy matplotlib tqdm

In [ ]:
# 1. Imports y configuración general

from pathlib import Path
import json
import time
import math
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageOps
from tqdm import tqdm

pd.set_option("display.max_columns", 200)

SEED = 42
np.random.seed(SEED)

print("OpenCV:", cv2.__version__)

In [ ]:
# 2. Montar Google Drive


from google.colab import drive

if Path("/content/drive/MyDrive").exists():
    print("Google Drive ya está montado.")
else:
    drive.mount("/content/drive")

In [ ]:
# 3. Rutas del proyecto

PROJECT_ROOT = Path("/content/drive/MyDrive/proyecto_integrador_v2")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"

if CONFIG_PATH.exists():
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
else:
    config = {
        "image_size": 224,
        "project_name": "proyecto_integrador_v2"
    }

RAW_DATA_PATH = PROJECT_ROOT / "raw_data"
RAW_IMAGES_PATH = RAW_DATA_PATH / "images"
LOST_REPORTS_PATH = RAW_DATA_PATH / "lost_reports"
FOUND_REPORTS_PATH = RAW_DATA_PATH / "found_reports"
IDENTITY_TEST_PATH = RAW_DATA_PATH / "identity_test"

CURATED_DATA_PATH = PROJECT_ROOT / "curated_data"
CURATED_IMAGES_PATH = CURATED_DATA_PATH / "images_curated"
QUALITY_REPORTS_PATH = CURATED_DATA_PATH / "quality_reports"

REPORTS_PATH = PROJECT_ROOT / "reports"
FIGURES_PATH = REPORTS_PATH / "figures"
TABLES_PATH = REPORTS_PATH / "tables"

for p in [CURATED_IMAGES_PATH, QUALITY_REPORTS_PATH, REPORTS_PATH, FIGURES_PATH, TABLES_PATH]:
    p.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = int(config.get("image_size", 224))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TARGET_SIZE:", TARGET_SIZE)

## Fuentes de imágenes

El notebook acepta distintas fuentes porque el proyecto puede trabajar con un dataset base, reportes de perros perdidos, reportes de perros encontrados y carpetas de evaluación donde varias fotos pertenecen al mismo perro.

Cada imagen conservará su `source_type` para que los pasos siguientes puedan diferenciar entre imágenes base, perdidas, encontradas o de evaluación.

In [ ]:
# 4. Buscar imágenes en raw_data

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

source_roots = [
    ("base_images", RAW_IMAGES_PATH),
    ("lost_report", LOST_REPORTS_PATH),
    ("found_report", FOUND_REPORTS_PATH),
    ("identity_test", IDENTITY_TEST_PATH),
]

image_records = []

for source_type, root in source_roots:
    if not root.exists():
        print("No existe:", root)
        continue

    for img_path in sorted(root.rglob("*")):
        if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTENSIONS:
            relative_path = img_path.relative_to(RAW_DATA_PATH)
            parts = relative_path.parts

            dog_id = None
            report_id = None

            if source_type == "identity_test" and len(parts) >= 2:
                dog_id = parts[1]
            elif source_type == "lost_report" and len(parts) >= 2:
                dog_id = parts[1]
                report_id = parts[1]
            elif source_type == "found_report" and len(parts) >= 2:
                report_id = parts[1]

            image_records.append({
                "image_path": str(img_path),
                "relative_path": str(relative_path),
                "source_type": source_type,
                "dog_id": dog_id,
                "report_id": report_id,
                "filename": img_path.name,
                "extension": img_path.suffix.lower()
            })

raw_images_df = pd.DataFrame(image_records)

print("Total de imágenes encontradas:", len(raw_images_df))
display(raw_images_df.head())

In [ ]:
# 5. Funciones de lectura y métricas de calidad visual

def read_image_rgb(image_path):
    """Lee una imagen y la regresa en RGB. Corrige orientación EXIF cuando es posible."""
    try:
        img = Image.open(image_path)
        img = ImageOps.exif_transpose(img)
        img = img.convert("RGB")
        return np.array(img)
    except Exception:
        return None


def compute_colorfulness(rgb):
    """Calcula colorfulness usando la métrica de Hasler-Süsstrunk."""
    rgb = rgb.astype(np.float32)
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]

    rg = np.abs(r - g)
    yb = np.abs(0.5 * (r + g) - b)

    std_rg = np.std(rg)
    std_yb = np.std(yb)
    mean_rg = np.mean(rg)
    mean_yb = np.mean(yb)

    return float(np.sqrt(std_rg ** 2 + std_yb ** 2) + 0.3 * np.sqrt(mean_rg ** 2 + mean_yb ** 2))


def compute_quality_metrics(rgb):
    """Calcula métricas visuales interpretables para una imagen RGB."""
    h, w = rgb.shape[:2]

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)

    brightness = float(np.mean(gray))
    contrast = float(np.std(gray))
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    saturation = float(np.mean(hsv[:, :, 1]))
    colorfulness = compute_colorfulness(rgb)

    edges = cv2.Canny(gray, threshold1=80, threshold2=160)
    edge_density = float(np.mean(edges > 0))

    aspect_ratio = float(w / h) if h > 0 else np.nan

    return {
        "width": int(w),
        "height": int(h),
        "aspect_ratio": aspect_ratio,
        "brightness_mean": brightness,
        "contrast_std": contrast,
        "sharpness_laplacian_var": sharpness,
        "saturation_mean": saturation,
        "colorfulness": colorfulness,
        "edge_density": edge_density
    }


def classify_quality(row):
    """Clasifica calidad visual con criterios prácticos."""
    brightness = row["brightness_mean"]
    contrast = row["contrast_std"]
    sharpness = row["sharpness_laplacian_var"]
    edge_density = row["edge_density"]
    width = row["width"]
    height = row["height"]

    flags = []

    if width < 150 or height < 150:
        flags.append("low_resolution")

    if brightness < 45:
        flags.append("too_dark")
    elif brightness > 220:
        flags.append("too_bright")

    if contrast < 25:
        flags.append("low_contrast")

    if sharpness < 80:
        flags.append("very_blurry")
    elif sharpness < 200:
        flags.append("low_sharpness")

    if edge_density < 0.015:
        flags.append("low_edge_density")

    if len(flags) == 0:
        quality = "good"
    elif len(flags) <= 2 and "very_blurry" not in flags:
        quality = "acceptable"
    else:
        quality = "needs_review"

    return quality, "|".join(flags) if flags else "none"

## Curaduría ligera

Para este proyecto no conviene aplicar transformaciones agresivas. La versión curada conserva la imagen en RGB y aplica ajustes moderados solo cuando son necesarios.

Se evita modificar en exceso porque detalles como manchas, textura del pelaje, bordes de orejas o forma del hocico pueden ser importantes para reconocer al mismo perro.

In [ ]:
# 6. Funciones de curaduría ligera

def light_enhance_rgb(rgb):
    """Aplica mejora ligera y conservadora. Evita sobreprocesar."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    l2 = clahe.apply(l)

    lab2 = cv2.merge([l2, a, b])
    enhanced = cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)

    # Mezcla con original para no cambiar demasiado identidad visual.
    blended = cv2.addWeighted(rgb, 0.70, enhanced, 0.30, 0)

    return blended


def resize_with_padding(rgb, target_size=224, padding_mode="mean"):
    """Redimensiona manteniendo proporción y agrega padding.

    Para re-identificación, preservar proporciones es importante.
    """
    h, w = rgb.shape[:2]
    target_w = target_size
    target_h = target_size

    scale = min(target_w / w, target_h / h)

    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))

    resized = cv2.resize(rgb, (new_w, new_h), interpolation=cv2.INTER_AREA)

    if padding_mode == "mean":
        pad_color = rgb.reshape(-1, 3).mean(axis=0).astype(np.uint8).tolist()
    elif padding_mode == "gray":
        pad_color = [128, 128, 128]
    elif padding_mode == "black":
        pad_color = [0, 0, 0]
    elif padding_mode == "white":
        pad_color = [255, 255, 255]
    else:
        pad_color = [128, 128, 128]

    canvas = np.full((target_h, target_w, 3), pad_color, dtype=np.uint8)

    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

    return canvas


def curate_image(rgb, target_size=224, apply_light_enhancement=True, create_model_ready=True):
    """Genera versión curada.

    enhanced_rgb: imagen mejorada ligeramente con tamaño original.
    model_ready_rgb: imagen 224x224 con padding para modelos.
    """
    if apply_light_enhancement:
        enhanced_rgb = light_enhance_rgb(rgb)
    else:
        enhanced_rgb = rgb.copy()

    if create_model_ready:
        model_ready_rgb = resize_with_padding(
            enhanced_rgb,
            target_size=target_size,
            padding_mode="mean"
        )
    else:
        model_ready_rgb = None

    return enhanced_rgb, model_ready_rgb

In [ ]:
# 7. Ejecutar análisis de calidad y curaduría

quality_records = []
start = time.time()

APPLY_LIGHT_ENHANCEMENT = True
CREATE_MODEL_READY_IMAGES = True
SAVE_CURATED_IMAGES = True

for _, row in tqdm(raw_images_df.iterrows(), total=len(raw_images_df)):
    image_path = Path(row["image_path"])
    relative_path = Path(row["relative_path"])

    record = row.to_dict()
    record.update({
        "read_status": "pending",
        "read_error": None,
        "curated_image_path": None,
        "model_ready_image_path": None
    })

    try:
        rgb = read_image_rgb(image_path)

        if rgb is None:
            record["read_status"] = "error"
            record["read_error"] = "image_not_readable"
            quality_records.append(record)
            continue

        metrics = compute_quality_metrics(rgb)
        record.update(metrics)

        quality_label, quality_flags = classify_quality(record)
        record["quality_label"] = quality_label
        record["quality_flags"] = quality_flags
        record["read_status"] = "ok"

        if SAVE_CURATED_IMAGES:
            enhanced_rgb, model_ready_rgb = curate_image(
                rgb,
                target_size=TARGET_SIZE,
                apply_light_enhancement=APPLY_LIGHT_ENHANCEMENT,
                create_model_ready=CREATE_MODEL_READY_IMAGES
            )

            curated_output_path = CURATED_IMAGES_PATH / "enhanced" / relative_path
            curated_output_path.parent.mkdir(parents=True, exist_ok=True)
            Image.fromarray(enhanced_rgb).save(curated_output_path, quality=95)
            record["curated_image_path"] = str(curated_output_path)

            if model_ready_rgb is not None:
                model_ready_output_path = CURATED_IMAGES_PATH / "model_ready_224" / relative_path
                model_ready_output_path.parent.mkdir(parents=True, exist_ok=True)
                Image.fromarray(model_ready_rgb).save(model_ready_output_path, quality=95)
                record["model_ready_image_path"] = str(model_ready_output_path)

    except Exception as e:
        record["read_status"] = "error"
        record["read_error"] = str(e)

    quality_records.append(record)

quality_df = pd.DataFrame(quality_records)

elapsed = (time.time() - start) / 60

print("Procesamiento terminado.")
print("Tiempo:", round(elapsed, 2), "minutos")
print("Registros:", len(quality_df))

if "read_status" in quality_df.columns:
    print("Status:")
    print(quality_df["read_status"].value_counts(dropna=False))

display(quality_df.head())

In [ ]:
# 8. Guardar reportes principales

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

QUALITY_REPORT_PATH = QUALITY_REPORTS_PATH / "step01_image_quality_report.csv"
QUALITY_REPORT_TIMESTAMPED_PATH = QUALITY_REPORTS_PATH / f"step01_image_quality_report_{timestamp}.csv"

quality_df.to_csv(QUALITY_REPORT_PATH, index=False)
quality_df.to_csv(QUALITY_REPORT_TIMESTAMPED_PATH, index=False)

quality_df.to_csv(TABLES_PATH / "step01_image_quality_report.csv", index=False)

print("Reporte guardado en:", QUALITY_REPORT_PATH)
print("Reporte con timestamp guardado en:", QUALITY_REPORT_TIMESTAMPED_PATH)

In [ ]:
# 9. Indicadores finales del Paso 01

total_images = len(quality_df)
read_ok = int((quality_df["read_status"] == "ok").sum()) if "read_status" in quality_df.columns else 0
read_errors = int((quality_df["read_status"] == "error").sum()) if "read_status" in quality_df.columns else 0

if read_ok > 0:
    ok_df = quality_df[quality_df["read_status"] == "ok"].copy()

    quality_counts = ok_df["quality_label"].value_counts().to_dict()

    indicators = [
        {"section": "input", "indicator": "total_images_found", "value": total_images},
        {"section": "input", "indicator": "readable_images", "value": read_ok},
        {"section": "input", "indicator": "read_errors", "value": read_errors},
        {"section": "quality", "indicator": "good_images", "value": int(quality_counts.get("good", 0))},
        {"section": "quality", "indicator": "acceptable_images", "value": int(quality_counts.get("acceptable", 0))},
        {"section": "quality", "indicator": "needs_review_images", "value": int(quality_counts.get("needs_review", 0))},
        {"section": "quality", "indicator": "avg_brightness", "value": float(ok_df["brightness_mean"].mean())},
        {"section": "quality", "indicator": "avg_contrast", "value": float(ok_df["contrast_std"].mean())},
        {"section": "quality", "indicator": "avg_sharpness", "value": float(ok_df["sharpness_laplacian_var"].mean())},
        {"section": "quality", "indicator": "avg_colorfulness", "value": float(ok_df["colorfulness"].mean())},
        {"section": "quality", "indicator": "avg_edge_density", "value": float(ok_df["edge_density"].mean())},
        {"section": "curation", "indicator": "light_enhancement_applied", "value": APPLY_LIGHT_ENHANCEMENT},
        {"section": "curation", "indicator": "model_ready_images_created", "value": CREATE_MODEL_READY_IMAGES},
        {"section": "curation", "indicator": "target_size", "value": TARGET_SIZE},
    ]
else:
    ok_df = pd.DataFrame()
    indicators = [
        {"section": "input", "indicator": "total_images_found", "value": total_images},
        {"section": "input", "indicator": "readable_images", "value": read_ok},
        {"section": "input", "indicator": "read_errors", "value": read_errors},
    ]

step01_indicators_df = pd.DataFrame(indicators)

STEP01_INDICATORS_PATH = QUALITY_REPORTS_PATH / "step01_quality_indicators.csv"
step01_indicators_df.to_csv(STEP01_INDICATORS_PATH, index=False)
step01_indicators_df.to_csv(TABLES_PATH / "step01_quality_indicators.csv", index=False)

display(step01_indicators_df)
print("Indicadores guardados en:", STEP01_INDICATORS_PATH)

In [ ]:
# 10. Resumen por fuente

if len(quality_df) > 0 and "quality_label" in quality_df.columns:
    source_summary_df = (
        quality_df
        .groupby(["source_type", "read_status", "quality_label"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values(["source_type", "read_status", "quality_label"])
    )

    SOURCE_SUMMARY_PATH = QUALITY_REPORTS_PATH / "step01_source_quality_summary.csv"
    source_summary_df.to_csv(SOURCE_SUMMARY_PATH, index=False)

    display(source_summary_df)
    print("Resumen por fuente guardado en:", SOURCE_SUMMARY_PATH)
else:
    print("No hay imágenes para resumir.")

In [ ]:
# 11. Visualizaciones de métricas

if len(ok_df) > 0:
    metrics_to_plot = [
        "brightness_mean",
        "contrast_std",
        "sharpness_laplacian_var",
        "colorfulness",
        "edge_density"
    ]

    for metric in metrics_to_plot:
        plt.figure(figsize=(8, 4))
        plt.hist(ok_df[metric].dropna(), bins=40)
        plt.title(f"Distribución de {metric}")
        plt.xlabel(metric)
        plt.ylabel("Frecuencia")
        plt.tight_layout()

        fig_path = FIGURES_PATH / f"step01_distribution_{metric}.png"
        plt.savefig(fig_path, dpi=150)
        plt.show()

        print("Figura guardada:", fig_path)
else:
    print("No hay imágenes válidas para graficar.")

In [ ]:
# 12. Muestra visual de imágenes curadas

def show_image_samples(df, n=8):
    valid_df = df[df["read_status"] == "ok"].copy()

    if len(valid_df) == 0:
        print("No hay imágenes para mostrar.")
        return

    sample_df = valid_df.sample(min(n, len(valid_df)), random_state=SEED)

    cols = min(4, len(sample_df))
    rows = math.ceil(len(sample_df) / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, (_, row) in enumerate(sample_df.iterrows()):
        img_path = row.get("model_ready_image_path") or row.get("curated_image_path") or row["image_path"]

        try:
            img = Image.open(img_path).convert("RGB")
            plt.subplot(rows, cols, i + 1)
            plt.imshow(img)
            plt.title(f"{row['source_type']}\\n{row['quality_label']}", fontsize=9)
            plt.axis("off")
        except Exception as e:
            print("Error mostrando:", img_path, e)

    plt.tight_layout()
    sample_path = FIGURES_PATH / "step01_curated_samples.png"
    plt.savefig(sample_path, dpi=150)
    plt.show()

    print("Muestra guardada en:", sample_path)

show_image_samples(quality_df, n=8)

# Análisis de métricas del Paso 01

El Paso 01 permite evaluar si las imágenes son adecuadas para las etapas posteriores de detección, embeddings y búsqueda visual.

Las métricas principales son:

- **Brillo promedio**: ayuda a identificar imágenes oscuras o sobreexpuestas.
- **Contraste**: mide la separación entre regiones claras y oscuras.
- **Nitidez**: estima qué tan definida está la imagen usando la varianza del Laplaciano.
- **Colorfulness**: mide riqueza de color, útil para patrones de pelaje y manchas.
- **Densidad de bordes**: ayuda a estimar estructura visual, contornos y detalles.
- **Aspect ratio**: permite detectar imágenes con proporciones extremas.

Para re-identificación visual, estas métricas son importantes porque el sistema necesita conservar información fina del perro. Una imagen muy borrosa, oscura o sin estructura visual puede generar embeddings menos confiables.